# Export YOLO `.pt` to TensorRT `.engine`

This notebook is focused on one objective: export a trained YOLO checkpoint (`.pt`) to TensorRT (`.engine`) and quickly validate it.


## 1. What This Notebook Does

- Validates GPU/CUDA/TensorRT availability.
- Exports a YOLO `.pt` model to TensorRT `.engine`.
- Runs an optional sanity prediction.
- Includes an optional quick speed comparison (`.pt` vs `.engine`).


## 2. Environment Check

Run this first. If TensorRT is missing, install it in your environment before exporting.


In [9]:
from __future__ import annotations

from pathlib import Path
import os
import torch

CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD.parent if CWD.name == "notebooks" else CWD

print(f"Project root: {PROJECT_ROOT}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Torch version: {torch.__version__}")


Project root: /home/robiotec/Documents/Entrenamientos/Training
CUDA available: True
GPU: NVIDIA GeForce RTX 5090
Torch version: 2.10.0+cu128


In [10]:
# Optional: quick driver/GPU check
!nvidia-smi


Tue May 12 12:48:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.142                Driver Version: 580.142        CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5090        Off |   00000000:01:00.0  On |                  N/A |
| 80%   27C    P8             36W /  600W |    2064MiB /  32607MiB |      9%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [11]:
# TensorRT version check (required for .engine export/inference)
import tensorrt
print(f"TensorRT version: {tensorrt.__version__}")


TensorRT version: 10.15.1.29


## 3. Configure Paths and Export Settings

Edit only this cell for each export.


In [16]:
from ultralytics import YOLO

# Input model (.pt)
RUN_DIR = PROJECT_ROOT / "result_tenguel" / "caja"/"TENGEL_V3_aug__run_2026-05-12"
PT_MODEL_PATH = RUN_DIR / "weights" / "TENGEL_V3.pt"

# Dataset yaml is needed for some export options (especially INT8 workflows)
DATA_YAML_PATH = PROJECT_ROOT / "configs" / "caja.yaml"

# Export configuration
EXPORT_CFG = {
    "format": "engine",    # TensorRT
    "imgsz": 640,
    "device": 0,
    "dynamic": False,
    "half": True,           # FP16 when supported
    "int8": False,          # Set True only if you prepared calibration flow
    "batch": 1,
    # "workspace": 8,       # Optional TensorRT workspace in GB
    "data": str(DATA_YAML_PATH),
}

assert PT_MODEL_PATH.exists(), f"Model not found: {PT_MODEL_PATH}"
assert DATA_YAML_PATH.exists(), f"Data yaml not found: {DATA_YAML_PATH}"

print(f"PT model: {PT_MODEL_PATH}")
print(f"Data yaml: {DATA_YAML_PATH}")
print(f"Export cfg: {EXPORT_CFG}")


PT model: /home/robiotec/Documents/Entrenamientos/Training/result_tenguel/caja/TENGEL_V3_aug__run_2026-05-12/weights/TENGEL_V3.pt
Data yaml: /home/robiotec/Documents/Entrenamientos/Training/configs/caja.yaml
Export cfg: {'format': 'engine', 'imgsz': 640, 'device': 0, 'dynamic': False, 'half': True, 'int8': False, 'batch': 1, 'data': '/home/robiotec/Documents/Entrenamientos/Training/configs/caja.yaml'}


## 4. Export `.pt` -> `.engine`


In [17]:
model = YOLO(str(PT_MODEL_PATH))
export_result = model.export(**EXPORT_CFG)

print("Export output:", export_result)


Ultralytics 8.4.13 🚀 Python-3.10.19 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5090, 32103MiB)
YOLO26n summary (fused): 122 layers, 2,375,031 parameters, 0 gradients, 5.2 GFLOPs

PyTorch: starting from '/home/robiotec/Documents/Entrenamientos/Training/result_tenguel/caja/TENGEL_V3_aug__run_2026-05-12/weights/TENGEL_V3.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 6) (5.1 MB)

ONNX: starting export with onnx 1.20.1 opset 20...
ONNX: slimming with onnxslim 0.1.84...


/home/robiotec/Documents/Entrenamientos/Training/.venv/lib/python3.10/site-packages/torch/onnx/_internal/torchscript_exporter/symbolic_opset11.py:954: UserWarning: Exporting aten::index operator of advanced indexing in opset 20 is achieved by combination of multiple ONNX operators, including Reshape, Transpose, Concat, and Gather. If indices include negative values, the exported graph will produce incorrect results.
  return opset9.index(g, self, index)


ONNX: export success ✅ 0.7s, saved as '/home/robiotec/Documents/Entrenamientos/Training/result_tenguel/caja/TENGEL_V3_aug__run_2026-05-12/weights/TENGEL_V3.onnx' (9.4 MB)

TensorRT: starting export with TensorRT 10.15.1.29...
[05/12/2026-12:52:05] [TRT] [I] ----------------------------------------------------------------
[05/12/2026-12:52:05] [TRT] [I] Input filename:   /home/robiotec/Documents/Entrenamientos/Training/result_tenguel/caja/TENGEL_V3_aug__run_2026-05-12/weights/TENGEL_V3.onnx
[05/12/2026-12:52:05] [TRT] [I] ONNX IR version:  0.0.9
[05/12/2026-12:52:05] [TRT] [I] Opset version:    20
[05/12/2026-12:52:05] [TRT] [I] Producer name:    pytorch
[05/12/2026-12:52:05] [TRT] [I] Producer version: 2.10.0
[05/12/2026-12:52:05] [TRT] [I] Domain:           
[05/12/2026-12:52:05] [TRT] [I] Model version:    0
[05/12/2026-12:52:05] [TRT] [I] Doc string:       
[05/12/2026-12:52:05] [TRT] [I] ----------------------------------------------------------------
TensorRT: input "images" with 

## 5. Resolve Engine Path

Ultralytics usually writes the engine next to the `.pt` file with the same base name.


In [ ]:
ENGINE_PATH = PT_MODEL_PATH.with_suffix(".engine")
print(f"Expected engine path: {ENGINE_PATH}")
print(f"Engine exists: {ENGINE_PATH.exists()}")


## 6. Sanity Inference (Optional)

Use a folder of test images to verify the exported engine works.


In [ ]:
TEST_SOURCE = PROJECT_ROOT / "data" / "splits" / "TENGEL_SPLIT_V1" / "test" / "images"
print(f"Test source: {TEST_SOURCE}")
print(f"Test source exists: {TEST_SOURCE.exists()}")

# Uncomment to run inference with the exported engine
# engine_model = YOLO(str(ENGINE_PATH), task="detect")
# engine_pred = engine_model.predict(
#     source=str(TEST_SOURCE),
#     conf=0.25,
#     device=0,
#     save=True,
#     project=str(PROJECT_ROOT / "result" / "engine_checks"),
#     name="sanity_predict_tenguel_caja",
#     exist_ok=True,
# )


## 7. Quick PT vs Engine Speed Check (Optional)


In [ ]:
import time

# Uncomment to benchmark the same source with both formats
# pt_model = YOLO(str(PT_MODEL_PATH))
# engine_model = YOLO(str(ENGINE_PATH), task="detect")
#
# t0 = time.time()
# _ = pt_model.predict(source=str(TEST_SOURCE), conf=0.25, device=0, save=False, verbose=False)
# t1 = time.time()
# _ = engine_model.predict(source=str(TEST_SOURCE), conf=0.25, device=0, save=False, verbose=False)
# t2 = time.time()
#
# print(f"PT inference time: {t1 - t0:.2f}s")
# print(f"Engine inference time: {t2 - t1:.2f}s")


## 8. Notes and Troubleshooting

- If export fails, verify CUDA/TensorRT compatibility with your installed PyTorch and GPU driver.
- If `int8=True`, make sure your calibration/data setup is correct.
- If you move files after export, update `PT_MODEL_PATH` / `ENGINE_PATH` accordingly.
